# 01 · The heavy steps on Euler: QC, trimming, kallisto, STAR and RSEM
**Goal:** run the compute-heavy steps on ETH's cluster (**Euler**) and copy the small results back. **Input:** the 27 `.sra` runs and GENCODE M39 (Euler downloads both). **Output:** `results/kallisto/`, `results/rsem/`, `results/qc/`, `results/star/`. **Kernel:** Python 3 (laptop part) · **Time:** per run STAR 5–13 min, kallisto 4–13 min, RSEM 13–97 min; the laptop stays idle.

```
27 .sra --> FASTQ --> FastQC (raw reads)
                  \-> fastp (adapters, poly-G, low quality) --> kallisto --------> transcript estimates
                                                            \-> STAR --+--------> reads per gene (unique reads)
                                                                       \-> RSEM -> expected reads per gene (EM)
```
All methods get the **same trimmed reads** and annotation. RSEM takes STAR's alignments to transcripts and shares multi-mapping reads between isoforms by EM, so its counts are not whole numbers. Jobs are submitted from a *login node* with **SLURM** and run on *compute nodes*; an **array** is one job split into 27 tasks, one per run. STAR's index needs ~30 GB RAM, hence the cluster.

| Step | Script | Resources |
|---|---|---|
| Software environment (versions pinned) | `euler/00_setup.sh` | login node |
| Download genome, annotation, transcripts, 27 runs | `euler/01_download.sh` (in `tmux`) | ~55 GB |
| kallisto index | `euler/02b_kallisto_index.sh` | 1 core, 32 GB (8 GB was too little); about 11 min |
| RSEM reference | `euler/02c_rsem_reference.sh` | 4 cores, 16 GB |
| STAR index | `euler/02_star_index.sh` | 10 cores, 60 GB; 22 min |
| **Light array:** FastQC + fastp + kallisto | `euler/03_process_array.sh`, `STAGES=qc+kallisto` | 27 tasks, 8 cores × 4 GB; measured 4–13 min per run, peak 3–23.7 GB |
| **STAR array:** fastp + STAR | `euler/03_process_array.sh`, `STAGES=star` | 27 tasks, 8 cores × 6 GB; 5–13 min per run (STAR alone) |
| **RSEM array:** RSEM on STAR's alignments | `euler/03_process_array.sh`, `STAGES=rsem` | 27 tasks, 8 cores × 3 GB; 13–97 min per run; peak memory 6.0–14.8 GB over 13 measured runs (task i starts after STAR task i) |
| Check results | `euler/05_check.sh` | read-only: all 27 runs present for every method, numbers in the expected range |
| Pack results | `euler/04_collect.sh` | four tarballs: STAR 13 MB, kallisto ~0.6 GB, RSEM ~70 MB, QC ~40 MB |

`submit.sh` submits everything with the right dependencies (`RANGE=7-27 bash euler/submit.sh` resumes part of the STAR and RSEM arrays). **Finished stages are skipped**; to redo one, delete its output first. `common.sh` holds the paths (`$SCRATCH/rnaseq_aging/`, folders with `_M39`) and loads conda. The `#SBATCH` lines at the top of each script are SLURM job requests (cores, memory, time, log); to bash they are comments.

## Part A · On Euler (ETH network or VPN)
```bash
ssh <user>@euler.ethz.ch
git clone https://github.com/Jeffateth/RNAseq_aging         # first time; later: cd RNAseq_aging && git pull
bash RNAseq_aging/euler/00_setup.sh                          # environment with pinned tools; rerun after every pull that changes it
tmux new -s dl                                               # tmux keeps the download alive if you disconnect
bash RNAseq_aging/euler/01_download.sh                       # resumable; leave with Ctrl-b then d
bash RNAseq_aging/euler/submit.sh                            # references/indices, then the three arrays
squeue -u $USER                                              # PD = pending, R = running; empty = finished
bash RNAseq_aging/euler/05_check.sh                          # last line must read ALL OK
bash RNAseq_aging/euler/04_collect.sh                        # first line must read STAR 27/27 | kallisto 27/27 | RSEM 27/27 | FastQC 27/27
```
Then, in a terminal on **your laptop** (not Euler; scratch deletes files after 15 days, so copy promptly):
```bash
mkdir -p results/euler && scp '<user>@euler.ethz.ch:/cluster/scratch/<user>/rnaseq_aging/*_results.tar.gz' results/euler/
```

## Part B · On the laptop: unpack, check and load the results

In [1]:
import sys; sys.path.insert(0, '../scripts')
import helpers as h
runs = h.load_runs()
# unpack the four tarballs from results/euler/ into results/kallisto, results/rsem, results/qc and results/star/mapping (missing ones are reported)
h.unpack_euler()

unpacked kallisto_results.tar.gz -> results


unpacked rsem_results.tar.gz -> results


unpacked qc_results.tar.gz -> results


unpacked star_results.tar.gz -> results/star


### 1 · kallisto, RSEM and QC checks
**Expect:** all 27 runs downloaded, trimmed and quantified, with pseudoaligned shares of 65–72 %. `pairs_M` = million read pairs after trimming; `aligned_%` = share compatible with annotated transcripts (reads from introns, unspliced RNA or unannotated sequence are not counted and play no part later); `q30_%` = bases with error rate ≤ 1 in 1,000; `dup_%` = fastp's duplicate estimate (3–8 %). Each tool's rates follow its own rules (section 3), so none is a quality score.

In [2]:
# one row per run: is it quantified and trimmed? raises an error if runs were processed with different trimming settings
h.check_consistent(runs, True)
h.status(runs)

consistent: all quantified runs used trimmed=True


,group,quantified,trimmed
SRR,,,
SRR21355401,aging 3mo,True,True
SRR21355396,aging 3mo,True,True
SRR21355392,aging 3mo,True,True
SRR21355388,aging 3mo,True,True
SRR21355398,aging 18mo,True,True
SRR21355394,aging 18mo,True,True
SRR21355390,aging 18mo,True,True
SRR21355386,aging 18mo,True,True
SRR21355397,aging 21mo,True,True


In [3]:
# here we run multiqc locally on the FastQC reports of the raw reads
# , and writes one HTML report, shows quality before trimming
print('report:', h.multiqc())

# reads pairs_M (number of read pairs in million after fastp trimming) and aligned_% (the share of those pairs that kallisto could assign to at least one transcript) from kallisto results and
# unassigned read pairs could come from introns or unannotated seqeunce or quality issues
# reads q30% (share of Q30 bases after filtering) and dup_% (duplicate estimate) from fastp's reports after trimming
# display the table in the notebook
h.qc_summary(runs) 

report: results/qc/multiqc_raw.html


,pairs_M,aligned_%,q30_%,dup_%,group
SRR21355401,14.9,65.8,89.8,7.6,aging 3mo
SRR21355396,25.7,67.6,87.4,6.9,aging 3mo
SRR21355392,16.1,67.6,88.2,6.6,aging 3mo
SRR21355388,16.9,67.9,88.5,7.2,aging 3mo
SRR21355398,6.4,68.1,86.5,3.3,aging 18mo
SRR21355394,7.8,67.9,86.2,2.9,aging 18mo
SRR21355390,28.2,65.1,88.7,8.0,aging 18mo
SRR21355386,9.3,68.1,86.1,3.3,aging 18mo
SRR21355397,14.2,67.0,88.5,5.6,aging 21mo
SRR21355393,9.8,70.1,87.7,3.8,aging 21mo


In [4]:
# reads RSEM's results
# genes: number of genes in the RSEM reference (78,289)
# genes_with_reads: genes with an expected count above 0 in that run (24,000-36,000)
# counts_M: total expected counts of the run in million (3-17);
# "expected count" because a read that fits several genes
# can't be assigned to one gene. RSEM splits it between them
# by estimated likelihood, so a gene can get a non-whole number
# count like 2.37.
h.check_rsem(runs)

,genes,genes_with_reads,counts_M,group
SRR21355401,78289,28109,7.38,aging 3mo
SRR21355396,78289,30123,12.91,aging 3mo
SRR21355392,78289,28512,8.19,aging 3mo
SRR21355388,78289,27379,8.49,aging 3mo
SRR21355398,78289,23972,3.24,aging 18mo
SRR21355394,78289,24803,3.98,aging 18mo
SRR21355390,78289,30545,13.31,aging 18mo
SRR21355386,78289,25206,4.79,aging 18mo
SRR21355397,78289,27648,7.29,aging 21mo
SRR21355393,78289,26552,5.38,aging 21mo


### 2 · STAR counts and mapping statistics
- `frac_read1_same_strand` ≈ 0.5: the libraries are **unstranded**, so the unstranded column is the right one.

In [5]:
# unique_%: share of read pairs mapped to one place in the genome
# multi_%: share of read pairs mapped to several places in the genome
# STAR_assigned_to_gene_%: share of read pairs that STAR counted in a gene (reads in introns or between genes are not counted)

counts, qc = h.load_star(runs)
print(counts.shape, 'genes x runs')
qc

(78289, 27) genes x runs


,input_pairs_M,unique_%,multi_%,STAR_assigned_to_gene_%,frac_read1_same_strand,group
SRR,,,,,,
SRR21355401,14.88,79.56,7.95,48.98,0.5,aging 3mo
SRR21355396,25.69,79.41,8.61,49.90,0.5,aging 3mo
SRR21355392,16.08,79.73,8.23,50.18,0.5,aging 3mo
SRR21355388,16.85,78.96,8.54,49.88,0.5,aging 3mo
SRR21355398,6.39,78.44,8.91,49.92,0.5,aging 18mo
SRR21355394,7.82,81.58,7.42,51.28,0.5,aging 18mo
SRR21355390,28.19,77.93,9.53,47.02,0.5,aging 18mo
SRR21355386,9.33,81.07,7.51,51.37,0.5,aging 18mo
SRR21355397,14.17,80.72,7.89,50.47,0.5,aging 21mo


In [6]:
# safety checks (27 runs; unstranded), then save the two tables that notebooks 04 and 05 read
assert counts.shape[1] == 27
assert qc.frac_read1_same_strand.between(0.4, 0.6).all(), 'library looks stranded: use another ReadsPerGene column'
counts.to_csv(h.STAR / 'counts_unstranded.csv.gz')
qc.to_csv(h.STAR / 'star_qc.csv')
print('saved results/star/counts_unstranded.csv.gz')

saved results/star/counts_unstranded.csv.gz


### 3 · Mapping rates: STAR vs kallisto
The rates have **different definitions**, so they are not directly comparable: `unique_%` counts pairs mapped to one place anywhere in the genome (also in introns); `STAR_assigned_to_gene_%` only those counted in a gene (multi-mappers dropped); kallisto's share counts pairs compatible with annotated transcripts (multi-mappers included). Expect `unique_%` clearly above the other two; each should be similar across runs. None is a quality score.

In [7]:
# STAR's three rates next to kallisto's pseudoaligned rate, per run (different definitions, see above)
k = h.qc_summary(runs)[['aligned_%']].rename(columns={'aligned_%': 'kallisto_pseudoaligned_%'})
qc[['unique_%', 'multi_%', 'STAR_assigned_to_gene_%']].join(k).join(runs['group'])

,unique_%,multi_%,STAR_assigned_to_gene_%,kallisto_pseudoaligned_%,group
SRR,,,,,
SRR21355401,79.56,7.95,48.98,65.8,aging 3mo
SRR21355396,79.41,8.61,49.90,67.6,aging 3mo
SRR21355392,79.73,8.23,50.18,67.6,aging 3mo
SRR21355388,78.96,8.54,49.88,67.9,aging 3mo
SRR21355398,78.44,8.91,49.92,68.1,aging 18mo
SRR21355394,81.58,7.42,51.28,67.9,aging 18mo
SRR21355390,77.93,9.53,47.02,65.1,aging 18mo
SRR21355386,81.07,7.51,51.37,68.1,aging 18mo
SRR21355397,80.72,7.89,50.47,67.0,aging 21mo
